# Emissions de CO₂ : scénarios courant vs déclarés

Comparaison des trajectoires d’émissions totales (toutes sources) en scénarios **courant** vs **stated policies** et lien avec la croissance des usages IA.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

root = Path('data/preproc')
df = pd.read_excel(root / 'total_co2_emissions.xlsx')

# Mise au format long
melt = df.melt(id_vars='Region', var_name='horizon', value_name='MtCO2')
# On sépare le nom du scénario
melt[['year', 'scenario']] = melt['horizon'].str.extract(r'(\d{4})_(.*)')

# Focus 2035/2050 et agrégation monde (somme si plusieurs régions)
focus = melt[melt['year'].isin(['2035', '2050'])]
global_totals = focus.groupby(['year', 'scenario'], as_index=False)['MtCO2'].sum()

ax = sns.barplot(data=global_totals, x='year', y='MtCO2', hue='scenario', palette='magma')
ax.set_title('Émissions globales: scénario courant vs stated policies')
ax.set_ylabel('Mt CO₂')
plt.tight_layout()
plt.show()


In [ ]:
# Ratio simple: comparer l’ordre de grandeur des émissions aux besoins data centers
import pandas as pd
from pathlib import Path

root = Path('data/preproc')
dc = pd.read_excel(root / 'data_center_electricity_demand_TWh.xlsx')
# Intensité carbone implicite pour donner un ordre de grandeur (hypothèse simplifiée 0.35 kg CO2/kWh)
INTENSITE = 0.35  # kg CO2 par kWh

est = dc[['Year', 'Total']].copy()
est['MtCO2_equiv'] = est['Total'] * 1e9 * INTENSITE / 1e6  # TWh -> kWh ; kg -> Mt
est.head()


In [ ]:
ax = sns.lineplot(data=est, x='Year', y='MtCO2_equiv', marker='o', label='Data centers (hypothèse 0.35 kg/kWh)')
ax.set_ylabel('Mt CO₂ équivalent')
ax.set_title('Ordre de grandeur des émissions associées aux data centers')
plt.tight_layout()
plt.show()


## Lecture rapide
- Les scénarios “current” sont nettement plus émissifs que les “stated policies” en 2035/2050 : l’écart illustre le besoin d’actions supplémentaires.
- Même avec une intensité carbone moyenne (0,35 kgCO₂/kWh), les data centers pourraient ajouter des centaines de MtCO₂ si le mix n’est pas fortement décarboné.
- Sans prédiction fine, cette comparaison souligne l’urgence d’une électricité décarbonée pour absorber l’IA.
